# 07 Ensemble with LUNAR v2

In [ ]:

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.append("../src")
from data import make_optuna_subsample, make_final_subsample
from metrics import find_best_f1_threshold, evaluate_scores, minmax_scale_scores
from results import build_experiment_record, save_record_json, get_memory_mb
from ensemble_utils import (
    tune_if, tune_lof, tune_dbscan, tune_ocsvm,
    score_if, score_lof, score_dbscan, score_ocsvm,
    tune_meta_fusion, apply_meta_fusion,
)

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS = ["CICIDS", "UNSW_NB15"]
SEED = 29
N_TRIALS = 200
META_TRIALS = 60
FUSION_STRATEGIES = ["mean", "max", "weighted", "rank_mean", "stacking_lr"]
DATASET_VERSION = "v1"
PREPROCESSING_VERSION = "v1"
SPLIT_METHOD = "stratified_train_val_test_fixed_seed"
RUN_CONFIGS = [
    dict(run_index=1, n_train_opt=7000,  n_val_opt=3000,  n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run1_small_opt_sample"),
    dict(run_index=2, n_train_opt=21000, n_val_opt=9000,  n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run2_medium_opt_sample"),
    dict(run_index=3, n_train_opt=35000, n_val_opt=15000, n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run3_large_opt_sample"),
]

import torch
sys.path.append("../external/LUNAR")
import LUNAR
import variables as var

MODEL_TYPE = "Ensemble_with_LUNAR_v2"
BASE_MODELS = ["LUNAR", "IF", "LOF", "DBSCAN", "OCSVM"]
SAMPLE_TYPES = ["UNIFORM", "SUBSPACE", "MIXED"]


def tune_lunar(train_x, train_y, val_x, val_y, dataset, seed, n_trials, results_dir):
    from optuna_utils import run_study
    from sklearn.metrics import roc_auc_score
    def objective(trial):
        params = {
            "k": trial.suggest_int("k", 5, 150, log=True),
            "samples": trial.suggest_categorical("samples", SAMPLE_TYPES),
            "lr": trial.suggest_float("lr", 1e-4, 1e-1, log=True),
            "wd": trial.suggest_float("wd", 1e-4, 1.0, log=True),
            "epsilon": trial.suggest_float("epsilon", 0.01, 0.5),
            "proportion": trial.suggest_int("proportion", 1, 2),
            "n_epochs": trial.suggest_int("n_epochs", 50, 300, step=25),
        }
        var.lr = params["lr"]; var.wd = params["wd"]; var.epsilon = params["epsilon"]
        var.proportion = params["proportion"]; var.n_epochs = params["n_epochs"]
        out = LUNAR.run(train_x, train_y, val_x, val_y, val_x, val_y, dataset, seed, params["k"], params["samples"], train_new_model=True)
        return roc_auc_score(val_y, out.numpy())
    return run_study(objective, f"LUNAR_tmp_{dataset}", seed, n_trials, results_dir=results_dir).best_params


def score_lunar(params, dataset, train_x, train_y, val_x, val_y, test_x, test_y):
    var.lr = params["lr"]; var.wd = params["wd"]; var.epsilon = params["epsilon"]
    var.proportion = params["proportion"]; var.n_epochs = params["n_epochs"]
    original_device = var.device; var.device = torch.device("cpu")
    t0 = time.time(); out_val = LUNAR.run(train_x, train_y, val_x, val_y, val_x, val_y, dataset, SEED, params["k"], params["samples"], train_new_model=True); tr = time.time() - t0
    t1 = time.time(); out_test = LUNAR.run(train_x, train_y, val_x, val_y, test_x, test_y, dataset, SEED, params["k"], params["samples"], train_new_model=True); inf = time.time() - t1
    var.device = original_device
    return minmax_scale_scores(out_val.numpy()), minmax_scale_scores(out_test.numpy()), tr, inf

records = []
for dataset in DATASETS:
    for run_cfg in RUN_CONFIGS:
        opt_train_x, opt_train_y, opt_val_x, opt_val_y = make_optuna_subsample(dataset, SEED, run_cfg["n_train_opt"], run_cfg["n_val_opt"])
        best_params = {"LUNAR": tune_lunar(opt_train_x, opt_train_y, opt_val_x, opt_val_y, dataset, SEED, N_TRIALS, RESULTS_DIR)}
        best_params.update({m: tuner(opt_train_x, opt_val_x, opt_val_y, SEED, N_TRIALS, RESULTS_DIR) for m, tuner in [("IF", tune_if), ("LOF", tune_lof), ("DBSCAN", tune_dbscan), ("OCSVM", tune_ocsvm)]})
        train_x, train_y, val_x, val_y, test_x, test_y = make_final_subsample(dataset, SEED, run_cfg["n_train_final"], run_cfg["n_val_final"], run_cfg["n_test_final"], max_nodes_budget=50_000_000, k=best_params["LUNAR"]["k"])
        val_cols, test_cols, runt_train, runt_inf = [], [], 0.0, 0.0
        for model_name in BASE_MODELS:
            if model_name == "LUNAR":
                v, t, tr, inf = score_lunar(best_params[model_name], dataset, train_x, train_y, val_x, val_y, test_x, test_y)
            elif model_name == "IF":
                v, t, tr, inf = score_if(best_params[model_name], train_x, val_x, test_x, SEED)
            elif model_name == "LOF":
                v, t, tr, inf = score_lof(best_params[model_name], train_x, val_x, test_x)
            elif model_name == "DBSCAN":
                v, t, tr, inf = score_dbscan(best_params[model_name], train_x, val_x, test_x)
            else:
                v, t, tr, inf = score_ocsvm(best_params[model_name], train_x, val_x, test_x)
            val_cols.append(v); test_cols.append(t); runt_train += tr; runt_inf += inf
        val_matrix = np.column_stack(val_cols)
        test_matrix = np.column_stack(test_cols)
        best_meta = tune_meta_fusion(val_matrix, val_y, SEED, META_TRIALS, RESULTS_DIR, FUSION_STRATEGIES)
        fused_val, fused_test = apply_meta_fusion(best_meta, val_matrix, test_matrix, val_y, SEED)
        thr, *_ = find_best_f1_threshold(val_y, fused_val)
        rec = build_experiment_record(dataset, DATASET_VERSION, SPLIT_METHOD, SEED, PREPROCESSING_VERSION, MODEL_TYPE, best_meta["fusion_strategy"], {"base_models": BASE_MODELS, "best_params": best_params, "meta": best_meta}, thr, fused_test, test_y, runt_train, runt_inf, get_memory_mb(), run_cfg["notes"])
        save_record_json(rec, RESULTS_DIR, run_cfg["run_index"], MODEL_TYPE, dataset)
        records.append(rec)

pd.DataFrame(records)
